# ASTR 457: Foundations of Data Science in Astronomy

**Fall 2026, University of Illinois Urbana-Champaign**

Prof. Gautham Narayan | TA: Abha Vishwakarma

Thu Oct 15, 2026 - Day 15: Visualization as verification: plots that catch broken models (on Zoom)

<img src="images/uiuc_logo.png" alt="University of Illinois Urbana-Champaign wordmark" width="220" style="display:block;margin:0 auto;">

## Recap

On Zoom at go.illinois.edu/gsn - ask in chat or unmute. How did the midterm go? One word in chat.

<img src="images/day14_hmc_vs_metropolis.gif" alt="Animation from Day 14: HMC points jump along a narrow tilted ellipse while Metropolis points creep from one end" style="display:block;margin:0 auto;max-height:175px">

* HMC's points jump along the ellipse, while Metropolis's points inch from one end
* Day 14's three checks:
    * the chain samples $p$, the posterior (detailed balance)
    * chains from overdispersed starts agree ($\hat R$)
    * there are enough independent samples (ESS)
* today we'll look at trace plots and $\tau$ (the autocorrelation time) on our own white-dwarf chains

**MCMC can give us very precise, but very wrong inferences, if the model itself is inadequate.**

## Today's plan

* today's question: how do you know a model is broken before a referee tells you?
* look at the data: 13 datasets, too many points, Gaia's HR diagram, Anscombe
* look at the residuals and the chains: a line fit to a curve, our white dwarfs, posterior predictive checks
* plots that fool you: Venus, K2-18 b, LK-99, log axes, color maps
* visualizing to communicate: what goes in, ambiguity, distortion, color for readers, small multiples, DESI, Tufte's rules. Tuesday continues with communication

In [ ]:
# Presenter setup (a RISE "skip" cell: it runs, but is not a slide).
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interactive_output, Dropdown, HBox, VBox, Layout
from IPython.display import display
plt.rcParams.update({'font.size': 13, 'legend.fontsize': 12})

def controls_right(fn, **widgets):
    # the plot on the left, its controls stacked on the right (the Day 12 layout)
    for w in widgets.values():
        w.layout = Layout(width='230px'); w.style = {'description_width': '75px'}
    display(HBox([interactive_output(fn, widgets), VBox(list(widgets.values()), layout=Layout(justify_content='center', width='240px'))]))

In [ ]:
# The three animations and the Anscombe residual figure in this deck, rebuilt only when missing (a "skip" cell)
import os
from matplotlib.animation import FuncAnimation, PillowWriter
from scipy import stats

def make_datasaurus_gif(path='images/day15_datasaurus_dozen.gif', data='data/thirteen_datasets.csv'):
    # Matejka & Fitzmaurice (2017): 13 datasets with the same means, spreads and correlation
    d = np.genfromtxt(data, delimiter=',', names=True)
    sets = [d[d['idx'] == k] for k in np.unique(d['idx'])]
    order = [2] + [k for k in range(len(sets)) if k != 2]          # the dinosaur (set 2) first
    fig, ax = plt.subplots(figsize=(6.4, 5.2))
    def draw(f):
        s = sets[order[f]]
        ax.clear()
        ax.plot(s['x'], s['y'], 'o', ms=5, color='C0')
        ax.set_xlim(0, 110); ax.set_ylim(-5, 105); ax.set_xlabel('$x$'); ax.set_ylabel('$y$')
        r = np.corrcoef(s['x'], s['y'])[0, 1]
        ax.set_title(f"mean x {s['x'].mean():.1f}, mean y {s['y'].mean():.1f}\n"
                     f"sd x {s['x'].std(ddof=1):.1f}, sd y {s['y'].std(ddof=1):.1f}, r = {r:.1f}", fontsize=13)
    draw(0); plt.tight_layout()
    FuncAnimation(fig, draw, frames=len(order)).save(path, writer=PillowWriter(fps=0.8)); plt.close(fig)

def make_residual_gif(path='images/day15_wrong_model_residuals.gif', n=40, sigma=1.0, seed=15):
    # a straight line fit to data with growing curvature: chi2/dof drifts up slowly, the residuals bend at once
    rng = np.random.default_rng(seed)
    x = np.linspace(0, 10, n); noise = rng.normal(0, sigma, n)
    curv = np.linspace(0, 0.09, 15)
    fig, (a1, a2) = plt.subplots(2, 1, figsize=(7.4, 5.8), sharex=True, gridspec_kw=dict(height_ratios=[2, 1.3]))
    def draw(f):
        y = 1 + 0.6 * x + curv[f] * (x - 5)**2 + noise
        p = np.polyfit(x, y, 1); res = (y - np.polyval(p, x)) / sigma
        chi2 = np.sum(res**2); dof = n - 2; pval = stats.chi2.sf(chi2, dof)
        a1.clear(); a2.clear()
        a1.errorbar(x, y, sigma, fmt='o', ms=4, color='0.3', elinewidth=0.8, label='data')
        a1.plot(x, 1 + 0.6 * x + curv[f] * (x - 5)**2, 'k--', lw=1.5, label='true curve')
        a1.plot(x, np.polyval(p, x), 'C3', lw=2.2, label='best straight line')
        a1.set_ylabel('$y$'); a1.legend(frameon=False, loc='upper left', ncol=3, fontsize=11)
        a1.set_title(rf'curvature $a$ = {curv[f]:.3f}:   $\chi^2/\nu$ = {chi2 / dof:.2f},  p = {pval:.2f}', fontsize=13)
        a2.axhline(0, color='C3', lw=1.5)
        a2.plot(x, res, 'o', ms=5, color='C0', label='residual of each point')
        xb, rb = x.reshape(5, 8).mean(1), res.reshape(5, 8).mean(1)
        a2.errorbar(xb, rb, 1 / np.sqrt(8), fmt='s', ms=7, color='C1', elinewidth=3, capsize=5, label='mean of each group of 8')
        a2.set_ylim(-3.5, 3.5); a2.set_xlabel('$x$'); a2.set_ylabel(r'residual / $\sigma$'); a2.legend(frameon=False, loc='upper center', ncol=2, fontsize=11)
    draw(0); plt.tight_layout()
    frames = [0] * 3 + list(range(len(curv))) + [len(curv) - 1] * 8          # hold the start and the end
    FuncAnimation(fig, draw, frames=frames).save(path, writer=PillowWriter(fps=1.5)); plt.close(fig)

def make_loglog_gif(path='images/day15_linear_to_loglog.gif', seed=15):
    # a broken power law with 5% scatter, morphing from linear axes to log-log
    rng = np.random.default_rng(seed)
    x = np.logspace(0, 2.3, 60)
    y = np.where(x < 20, (x / 20)**-1.5, (x / 20)**-3.0) * 100 * rng.lognormal(0, 0.05, x.size)
    lx, ly = np.log10(x), np.log10(y)
    def norm(v): return (v - v.min()) / (v.max() - v.min())
    fig, ax = plt.subplots(figsize=(6.6, 4.6))
    ts = np.r_[np.zeros(6), np.linspace(0, 1, 16), np.ones(10)]
    def draw(f):
        t = ts[f]
        ax.clear()
        if t == 1:                                                 # the end: real log-log axes and the two planted slopes
            ax.loglog(x, y, 'o', ms=5, color='C0')
            for lo, hi, a in [(1, 20, -1.5), (20, 200, -3.0)]:
                xx = np.logspace(np.log10(lo), np.log10(hi), 20); ax.loglog(xx, 100 * (xx / 20)**a, 'C3', lw=2)
            ax.text(1.5, 120, 'slope -1.5', color='C3', fontsize=13); ax.text(55, 30, 'slope -3', color='C3', fontsize=13)
            ax.set_xlabel('luminosity  (log scale)'); ax.set_ylabel('number per bin  (log scale)')
            ax.set_title('log-log: two straight lines, and a break', fontsize=13)
            return
        ax.plot((1 - t) * norm(x) + t * norm(lx), (1 - t) * norm(y) + t * norm(ly), 'o', ms=5, color='C0')
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_xlabel('luminosity' + ('  (log scale)' if t == 1 else '  (linear)' if t == 0 else ''))
        ax.set_ylabel('number per bin' + ('  (log scale)' if t == 1 else '  (linear)' if t == 0 else ''))
        ax.set_title('linear axes: one steep fall' if t == 0 else ('log-log: two straight lines, and a break' if t == 1 else ''), fontsize=13)
    draw(len(ts) - 1); plt.tight_layout()                      # lay out with the tick labels of the last frame
    FuncAnimation(fig, draw, frames=len(ts)).save(path, writer=PillowWriter(fps=4)); plt.close(fig)

def make_anscombe_residuals(path='images/day15_anscombe_residuals.png'):
    # Anscombe (1973): residuals of each set from the shared best-fit line y = 3 + 0.5x
    x13 = np.array([10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5.])
    sets = [(x13, [8.04, 6.95, 7.58, 8.81, 8.33, 9.96, 7.24, 4.26, 10.84, 4.82, 5.68]),
            (x13, [9.14, 8.14, 8.74, 8.77, 9.26, 8.10, 6.13, 3.10, 9.13, 7.26, 4.74]),
            (x13, [7.46, 6.77, 12.74, 7.11, 7.81, 8.84, 6.08, 5.39, 8.15, 6.42, 5.73]),
            (np.array([8, 8, 8, 8, 8, 8, 8, 19, 8, 8, 8.]), [6.58, 5.76, 7.71, 8.84, 8.47, 7.04, 5.25, 12.50, 5.56, 7.91, 6.89])]
    fig, axs = plt.subplots(1, 4, figsize=(12, 3.0), sharey=True)
    for k, (ax, (x, y)) in enumerate(zip(axs, sets)):
        ax.axhline(0, color='C3', lw=1.5); ax.plot(x, np.array(y) - (3 + 0.5 * x), 'o', color='C0', ms=7)
        ax.set_title(f'dataset {k + 1}', fontsize=13); ax.set_xlabel('$x$'); ax.set_xlim(3, 20)
    axs[0].set_ylabel('residual from\n$y = 3 + 0.5x$'); plt.tight_layout(); plt.savefig(path, dpi=110); plt.close(fig)

for _path, _make in [('images/day15_datasaurus_dozen.gif', make_datasaurus_gif),
                     ('images/day15_wrong_model_residuals.gif', make_residual_gif),
                     ('images/day15_linear_to_loglog.gif', make_loglog_gif),
                     ('images/day15_anscombe_residuals.png', make_anscombe_residuals)]:
    if not os.path.exists(_path):
        _make()
print('animations ready')

## This week

* the **midterm** is in - thank you. Defenses on it start Tuesday
* Abha's office hours: Friday, 1-2 pm, over Zoom
* **Quiz 3**: probably Tue Oct 20, on Bayes and MCMC (Days 12-14)
* your capstone topic, chosen with me, by Tue Oct 20
* next lab defense, Tue Oct 20 in person: **Andrew Steves**, on the midterm
* Tuesday is back in person, in Astronomy 134. Every date is in `ASSIGNMENTS.md`

## Your turn: 13 datasets

* `lecture/08/data/thirteen_datasets.csv` (`git pull` first), or read it straight from GitHub. Columns: `idx` (which dataset, 0-12), `x`, `y`

```python
import pandas as pd
d = pd.read_csv('https://raw.githubusercontent.com/gnarayan/ast457_2026_Fall/main/lecture/08/data/thirteen_datasets.csv')
d.groupby('idx').agg(['mean', 'std']).round(1)                           # means and spreads
d.groupby('idx')[['x', 'y']].apply(lambda g: g.x.corr(g.y)).round(2)    # correlation
```

<div style="font-size:1.15em; line-height:1.4; margin:0.6em 0; padding:0.5em 0.8em; border-left:4px solid #444; background:#f2f2f2;">

💬 3 minutes. From the numbers alone, will the 13 datasets look alike? Yes or no in chat. Then plot two: `plt.plot(d.x[d.idx == 2], d.y[d.idx == 2], 'o')`

</div>

**Start with summary statistics, then plot the data.**

## What your code prints

<img src="images/day15_thirteen_stats_table.png" alt="Table of the 13 datasets: every one has mean x 54.3, mean y 47.8, sd x 16.8, sd y 26.9 and correlation about -0.06" style="display:block;margin:0 auto;max-height:260px">

<div style="font-size:0.75em; text-align:center; color:#666;">Means, standard deviations and correlation of the 13 datasets in `thirteen_datasets.csv`, computed for this course.</div>

* all 13 agree to the first decimal place - so do they look alike? Plot them

**Same numbers. Now look at the data.**

## The Datasaurus

<img src="images/day15_datasaurus_dozen.gif" alt="Animation of thirteen scatter plots in turn, a dinosaur, a star, a circle, lines and crosses, with the same means, standard deviations and correlation printed above each" style="display:block;margin:0 auto;max-height:300px">

<div style="font-size:0.75em; text-align:center; color:#666;">Data: Matejka &amp; Fitzmaurice 2017, CHI, "Same Stats, Different Graphs", starting from the Datasaurus by Alberto Cairo; animation made for this course.</div>

* this might tempt you to think the datasets are largely similar. But let's look at the data
* the numbers in the title stay the same while the data change

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Moments alone do not fully characterize a distribution. LOOK AT YOUR DATA.**

</div>

## Too many points

<img src="images/sdss_25m_telescope.jpg" alt="The Sloan 2.5 m telescope at Apache Point Observatory, a white-shuttered telescope on a platform above forested hills" style="display:block;margin:0 auto;max-height:180px">

<div style="font-size:0.75em; text-align:center; color:#666;">The Sloan 2.5 m telescope at Apache Point Observatory. Credit: Fermilab Visual Media Services / Sloan Digital Sky Survey.</div>

* SDSS measured temperature and surface gravity for ~330,000 stars. Here: a random 150,000
* first try: one point per star, `plt.plot(teff, logg, 'k.', ls='None')`
* try each option in the dropdown: points, faint points, a 2D histogram, and contours with only the outliers as points (all four are in one figure in the notebook, if you're reading it without running it)

<div style="font-size:0.75em; text-align:center; color:#666;">Data: SDSS DR9 SEGUE Stellar Parameter Pipeline, via astroML (Ivezić et al. 2014). Example adapted from F. Bianco's DSPS visualization lecture (slides.com/federicabianco/dsps_09).</div>

In [ ]:
# 150,000 SDSS SSPP stars and four ways to draw them (a "skip" cell)
from scipy.ndimage import gaussian_filter
_sp = np.load('data/sdss_sspp_teff_logg.npz'); TEFF, LOGG = _sp['teff'], _sp['logg']
_H, _xe, _ye = np.histogram2d(TEFF, LOGG, bins=(120, 90), range=((4200, 8300), (0.5, 5.2)))
_Hs = gaussian_filter(_H, 1.0)
def plot_many(style='points'):
    fig, ax = plt.subplots(figsize=(7.2, 4.6))
    if style == 'points':
        ax.plot(TEFF, LOGG, 'k.', ms=2)
    elif style == 'faint points':
        ax.plot(TEFF, LOGG, 'k.', ms=1, alpha=0.06)
    elif style == '2D histogram':
        im = ax.pcolormesh(_xe, _ye, np.log10(_H.T + 1), cmap='viridis'); fig.colorbar(im, ax=ax, label='log$_{10}$(1 + stars per bin)')
    else:
        lev = np.percentile(_Hs[_Hs > 0], [50, 75, 90, 97, 99.5])
        ix = np.clip(np.digitize(TEFF, _xe) - 1, 0, len(_xe) - 2); iy = np.clip(np.digitize(LOGG, _ye) - 1, 0, len(_ye) - 2)
        out = _Hs[ix, iy] < lev[0]
        ax.plot(TEFF[out], LOGG[out], 'k.', ms=1.5)
        ax.contour(0.5 * (_xe[1:] + _xe[:-1]), 0.5 * (_ye[1:] + _ye[:-1]), _Hs.T, levels=lev, colors='k', linewidths=1)
    ax.set_xlim(8300, 4200); ax.set_ylim(5.2, 0.5)
    ax.set_xlabel('$T_{\\rm eff}$ (K)'); ax.set_ylabel('$\\log g$ (cgs)'); ax.set_title(style, fontsize=13)
    plt.tight_layout(); plt.show()

# a static copy of all four views, for anyone reading the notebook without running it
if not os.path.exists('images/day15_sspp_four_ways.png'):
    _show = plt.show; plt.show = lambda: None; _figs = []
    fig4, axs4 = plt.subplots(1, 4, figsize=(16, 3.8), sharey=True)
    for _ax, _st in zip(axs4, ['points', 'faint points', '2D histogram', 'contours + outliers']):
        plot_many(_st); _f = plt.gcf(); _f.canvas.draw()
        _ax.imshow(np.asarray(_f.canvas.buffer_rgba())); _ax.axis('off'); plt.close(_f)
    plt.figure(fig4.number); plt.tight_layout(); fig4.savefig('images/day15_sspp_four_ways.png', dpi=110, bbox_inches='tight'); plt.close(fig4)
    plt.show = _show


In [ ]:
# the same 150,000 stars, four ways
controls_right(plot_many, style=Dropdown(options=['points', 'faint points', '2D histogram', 'contours + outliers'], value='points', description='draw as'))

*Reading this without running the notebook? Here are all four views of the same 150,000 stars:*

<img src="images/day15_sspp_four_ways.png" alt="Four views of 150,000 SDSS stars, temperature against surface gravity: solid points saturated black, faint points, a 2D histogram showing the main sequence and giant branch, and contours with only the outliers drawn as points" style="display:block;margin:0 auto;max-width:100%">

## Too many points: what the plots showed

<div style="font-size:1.15em; line-height:1.4; margin:0.6em 0; padding:0.5em 0.8em; border-left:4px solid #444; background:#f2f2f2;">

💬 in "points", where are most of the stars? Then I switch to the histogram.

</div>

<img src="images/day15_sspp_four_ways.png" alt="Four views of 150,000 SDSS stars: solid points saturated black, faint points, a 2D histogram, and contours with outliers" style="display:block;margin:0 auto;max-height:150px">

<div style="font-size:0.75em; text-align:center; color:#666;">The four views of the widget, as static images.</div>

* solid points pile up: the densest region and a sparse one look equally black
* the histogram and contours show where the stars are: the main sequence (dwarfs, $\log g \approx 4$-4.5) and the giant branch, rising to low $\log g$ at cool temperatures. Keeping the outliers as points shows what is odd
* the hard edges at 4500 K and 8000 K are where the catalog was cut - hotter and cooler stars just aren't in this sample

**When points overlap, plot the density, and keep the outliers.**

## Gaia DR2: the HR diagram

<img src="images/lindegren2018_fig26_hrd_raw_vs_filtered.png" alt="Three Gaia DR2 color-magnitude diagrams of nearby stars: the first has many points between the main sequence and the white dwarfs; cuts on fit quality remove almost all of them in the second and third" style="display:block;margin:0 auto;max-height:210px">

<div style="font-size:0.75em; text-align:center; color:#666;">Lindegren et al. 2018, A&amp;A 616, A2 (Gaia DR2 astrometry), Fig. C.1.</div>

* a color-magnitude (HR) diagram from Gaia's positions and parallaxes (astrometry): stars within 100 pc, parallaxes measured to 10%
* a parallax that is too large puts a star too close. We then infer it is less luminous than it is, and it drops below the main sequence
* panel 1: points fill the gap between the main sequence and the white dwarfs (the cores of dead stars, bottom left)
* panel 2 cuts on astrometric fit quality, panel 3 also on a photometry flag. Almost all of them go away

**If stars land where no stars should be, something upstream is wrong. No substitute for visual inspection.**

## Anscombe's quartet

<img src="images/anscombe1973_quartet_wikimedia.png" alt="Four scatter plots with the same fitted line: a noisy linear cloud, a smooth curve, a tight line with one outlier, and a vertical stack with one far point" style="display:block;margin:0 auto;max-height:280px">

<div style="font-size:0.75em; text-align:center; color:#666;">Anscombe 1973, The American Statistician 27, 17. Rendering: Schutz / Avenue, Wikimedia Commons, CC BY-SA 3.0.</div>

* the same means, variances, correlation and best-fit line in all four
* <a href="https://seeing-theory.brown.edu/regression-analysis/index.html" target="_blank">launch Seeing Theory</a> (link also in chat): Regression, "Ordinary least squares", Anscombe's quartet. Drag the points and watch the fit

<div style="font-size:1.15em; line-height:1.4; margin:0.6em 0; padding:0.5em 0.8em; border-left:4px solid #444; background:#f2f2f2;">

💬 which of the four would you fit a straight line to? A number, 1-4, in chat.

</div>

**The best-fit line alone cannot tell you which dataset it suits.**

## Anscombe: the answer

<img src="images/day15_anscombe_residuals.png" alt="Four panels of residuals from the same best-fit line y = 3 + 0.5x for the four Anscombe datasets: random scatter for set 1, an arch for set 2, one large outlier for set 3, and a single point at x = 19 for set 4" style="display:block;margin:0 auto;max-height:300px">

<div style="font-size:0.75em; text-align:center; color:#666;">Anscombe 1973; residuals from the shared line $y = 3 + 0.5x$, drawn for this course.</div>

* only dataset 1 is a line plus noise. Its residuals scatter around zero
* dataset 2 is a curve (an arch), 3 a line with one outlier, 4 a single point setting the slope

**The residuals show what the best-fit line hides.**

## Goodness of fit

<img src="images/day15_residuals_frame0.png" alt="The first frame of the animation: data with error bars, the true curve dashed and the best straight line in red on top; residuals as blue dots and orange squares below" style="display:block;margin:0 auto;max-height:230px">

<div style="font-size:0.75em; text-align:center; color:#666;">The first frame ($a = 0$): the line is the right model here, and the orange squares sit on zero.</div>

* fake data: $y = 1 + 0.6x + a\,(x-5)^2$ plus Gaussian noise with $\sigma = 1$. The number $a$ sets how curved the data are
* the fit: a straight line, $y = mx + b$, by least squares. Once $a > 0$ it is the wrong model
* each frame: the same noise, a slightly larger $a$ (printed at the top), and a new best-fit line
* top panel: data (gray), the true curve (dashed), the best line (red)
* bottom panel: each point's residual, (data − line)/$\sigma$ (blue dots). Orange squares: the mean residual of each group of 8 neighboring points, with its error bar

## Residuals

<img src="images/day15_wrong_model_residuals.gif" alt="Animation: data with a growing curve, the true curve dashed and a straight-line fit on top; below, the residuals, whose binned means go high at the ends and low in the middle while chi-squared per degree of freedom rises to 1.38" style="display:block;margin:0 auto;max-height:340px">

* the orange squares go high, low, high: the line misses the curve
* at the largest curvature, $a = 0.09$, $\chi^2/\nu = 1.38$ and $p = 0.06$. With $p > 0.05$, a $\chi^2$ test still passes
* if the model is wrong, "best" might not be particularly revealing!

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**A $\chi^2$ test tells us whether the fit is acceptable, but not what is wrong with the model. Look at the residuals!**

</div>

## Our white dwarfs

* white dwarfs are the cores of dead Sun-like stars. We fit their hydrogen (Balmer) lines to get the temperature, $T_{\rm eff}$
* we use them as flux standards - stars whose known brightness calibrates other measurements. Their temperatures have to be right
<img src="images/narayan2019_wdmodel_j1206_mcmc.png" alt="Spectrum of the white dwarf SDSS J1206+0201 with deep hydrogen absorption lines and the full-fit model in red on top; residuals below" style="display:block;margin:0 auto;max-height:150px">

<div style="font-size:0.75em; text-align:center; color:#666;">The full fit (red) to the Gemini spectrum of SDSS J1206+0201: the deep dips are the hydrogen Balmer lines. WDmodel output, Narayan et al. 2019.</div>

* the quick fit treats each pixel's error as independent. The full fit models correlated noise between neighboring pixels, and adds the HST photometry

<div style="font-size:0.75em; text-align:center; color:#666;">Narayan et al. 2019, ApJS 241, 20, doi:10.3847/1538-4365/ab0557. The full sample: Axelrod et al. 2023, ApJ 951, 78.</div>

**Same spectrum, two noise models.**

## Quick fit vs full fit

<img src="images/narayan2019_wdmodel_j1206_residuals_quick_vs_full.png" alt="Residuals of two fits to the white dwarf SDSS J1206+0201: the quick fit leaves wiggles between 3800 and 4400 angstroms, the full fit leaves flat residuals" style="display:block;margin:0 auto;max-height:280px">

<div style="font-size:0.75em; text-align:center; color:#666;">WDmodel fits of SDSS J1206+0201 (Gemini spectrum), Narayan et al. 2019.</div>

* quick fit: $T_{\rm eff} = 21{,}723 \pm 66$ K, with residuals that wiggle at the Balmer lines
* full fit: $T_{\rm eff} = 23{,}434^{+456}_{-408}$ K. The quick fit is 1,711 K, or 26σ, below the full fit (!)

**The ±66 K error bar looked fine, but the residuals already showed the fit was wrong.**

## Parallel tempering

<img src="images/narayan2019_wdmodel_j1206_temperatures.png" alt="Histograms of the star temperature from the walkers at the four sampler temperature levels: level 1 is narrowest and level 4 is widest with a long tail" style="display:block;margin:0 auto;max-height:190px">

* I used emcee's parallel-tempering sampler: we run one set of walkers at $T = 1$ and three more sets at $T > 1$ (a sampler "temperature" - nothing to do with the star's $T_{\rm eff}$)
* at temperature $T$ we sample prior × likelihood$^{1/T}$. For $T > 1$ that is smoothed out. This allows chains to move among multiple peaks more easily
* the sets swap positions (a quarter of proposed swaps were accepted), so the $T = 1$ walkers reach regions they'd otherwise rarely propose. The figure: $T_{\rm eff}$ at each level - $T = 1$ is the narrowest, and only its samples are used for inference

**Of course, we're only actually interested in $T=1$...**

## The full fit: trace plots

<img src="images/narayan2019_wdmodel_j1206_trace.png" alt="Trace plots of temperature and surface gravity for 20 of 100 walkers: on the left the first 1,500 iterations, where the walkers start at the quick-fit temperature and climb; on the right all 50,000 iterations, flat around one value" style="display:block;margin:0 auto;max-height:300px">

<div style="font-size:0.75em; text-align:center; color:#666;">The SDSS J1206+0201 chain: emcee's parallel-tempering sampler, 4 temperatures × 100 walkers, 50,000 iterations, every 10th saved. Plotted for this course from the saved chain.</div>

* left: the walkers start at the quick fit, 21,723 K, and reach about 23,400 K within ~500 iterations. Conservatively, we might remove the first ~500 iterations based on this. This period is called the **burn-in**
* right: after that, is the chain stationary? Here, yes - no drift for 50,000 iterations, and no stuck walker

**The trace shows where the burn-in ends.**

## How many independent samples?

<img src="images/narayan2019_wdmodel_j1206_autocorr.png" alt="Two panels of estimated autocorrelation time against chain length for four parameters: from iteration 0 the estimates are large and drift, with distance near 140 iterations; with the first 2,000 iterations dropped all four sit flat near 13 iterations, far below the N over 50 line" style="display:block;margin:0 auto;max-height:215px">

* $\tau$, the autocorrelation time (Day 12): the lag you need before samples are nearly independent. ESS = walkers × iterations / $\tau$
* left: $\tau$ estimated without removing the burn-in - for the distance to the star (a fit parameter, it sets the brightness) it looks like 140 iterations
* right: drop the first 2,000 iterations (the trace settles by ~500; we drop more to be safe) and $\tau \approx 13$ iterations for every parameter - about one saved step, since we kept every 10th iteration
* 48,000 iterations is ≳3,700 $\tau$, far above emcee's 50 $\tau$ rule (i.e. run for at least 50 $\tau$), and ESS ≈ 370,000

**Estimate $\tau$ after the burn-in, and run for at least 50 $\tau$.**

## The full fit, as a corner plot

<img src="images/narayan2019_wdmodel_j1206_corner.png" alt="Corner plot of nine parameters from the full white-dwarf fit: several one-dimensional histograms are lopsided and several pairs are strongly correlated" style="display:block;margin:0 auto;max-height:340px">

<div style="font-size:0.75em; text-align:center; color:#666;">The same chain for SDSS J1206+0201 (cold walkers): temperature, surface gravity, dust, distance, resolution and the noise-model parameters.</div>

* several parameters are far from Gaussian, and some pairs are strongly correlated
* the quick fit's ±66 K came from an optimizer, and it would hide both

**Save your chains and plot them, so that people can decide whether you did sensible things.**

## Posterior predictive checks

* the idea is to generate data from the model using parameters drawn from the posterior, with the noise, and compare with what we observed (Day 12, Lab 05)
* here: Day 9's 166 Cepheids in NGC 4258 (Riess et al. 2011) and three models from Day 9's fits - a line with only the error bars, a line plus intrinsic scatter, and the mixture model with scatter

<img src="images/day15_ppc_r11_residuals.png" alt="Three panels of residual distributions: the real Cepheid residuals in black against 100 fake datasets from each model; the error-bars-only fakes are too narrow and too peaked, the other two models follow the black curve" style="display:block;margin:0 auto;max-height:175px">

<div style="font-size:0.75em; text-align:center; color:#666;">Black: the real residuals from the best line. Colored: 100 fake datasets per model. Data: Riess et al. 2011, ApJ 730, 119, Table 2 (NGC 4258); plotted for this course.</div>

* the error-bars-only fakes are too narrow: the real Cepheids scatter more than their error bars allow

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Does our data look anything like data generated from the model?**

</div>

## Posterior predictive check: a test statistic

<img src="images/day15_ppc_r11_teststat.png" alt="Three histograms of the number of stars more than 0.75 mag off the line in fake datasets, with a black line at the real value of 37: far to the right of the error-bars-only histogram, inside the other two" style="display:block;margin:0 auto;max-height:210px">

* here the test statistic is the number of stars more than 0.75 mag off the line (i.e. Riess et al.'s own rejection cut)
* real data: 37. Error bars only: about 15, and never 37 in 400 fake datasets. Scatter or mixture: about 42-44, and 37 sits inside both histograms
* the check rules out model 1, but it can't choose between models 2 and 3 - for that we need the evidence (Day 13)

**If the model is right, you expect to see something similar to what you did the first time.**

## Plots that fool you

<img src="images/day15_fool_montage.png" alt="Three figures from the next slides: the Venus phosphine spectrum, the K2-18 b transmission spectrum, and LK-99 resistivity" style="display:block;margin:0 auto;max-height:170px">

<div style="font-size:0.75em; text-align:center; color:#666;">Coming up: Greaves et al. 2020 (Venus), Madhusudhan et al. 2025 (K2-18 b), Lee et al. 2023 (LK-99).</div>

* so far, plots that catch a broken model. Now the reverse
* several passed a referee. A second look at the data caught each one
* for each one, ask what was plotted, what was fitted, and what was left out

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**If you do not look at your data, well, congratulations, you will fool yourself.**

</div>

## Phosphine on Venus

* phosphine ($\rm PH_3$): on Earth it comes from anaerobic life or from industry. Finding it on Venus could point to biology
* ALMA (Day 9) spectrum around the line; velocity 0 is where it should be. The baseline ripples, so the authors fitted a 12th-order polynomial, skipping the line region

<img src="images/greaves2020_fig2_venus_alma_ph3.png" alt="ALMA spectra of Venus around the phosphine line, showing an absorption dip at zero velocity" style="display:block;margin:0 auto;max-height:180px">

<div style="font-size:0.75em; text-align:center; color:#666;">Greaves et al. 2020, Nature Astronomy 5, 655 (arXiv:2009.06593), Fig. 2: ALMA spectra after a 12th-order polynomial baseline was removed.</div>

<div style="font-size:1.15em; line-height:1.4; margin:0.6em 0; padding:0.5em 0.8em; border-left:4px solid #444; background:#f2f2f2;">

💬 a 12th-order polynomial fitted around a gap: does it follow the wiggles or smooth them? One word in chat.

</div>

**The detection rests on how the authors removed the baseline.**

## Venus: the fit on other noise bumps

<img src="images/snellen2020_fig2_venus_fake_lines.png" alt="Six spectra from the same ALMA data after the same 12th-order fit, each showing an apparent absorption or emission line at a velocity where no line exists" style="display:block;margin:0 auto;max-height:215px">

<div style="font-size:0.75em; text-align:center; color:#666;">Snellen et al. 2020, A&amp;A 644, L2 (arXiv:2010.09761), Fig. 2, bottom rows: the same 12th-order fit applied at other velocities.</div>

* a 12th-order polynomial has 13 free coefficients - flexible enough to follow the ripples, and to dig a "line" into the gap it skipped
* look at the dips and bumps: five more "lines", above $10\sigma$, at velocities with no line
* with a 3rd-order baseline, the phosphine dip drops to about $2\sigma$
* Greaves et al. 2021 also found an ALMA calibration error and revised the abundance down, from about 20 to 1-4 parts per billion. It is still disputed

**"With four parameters I can fit an elephant, and with five I can make him wiggle his trunk."** - John von Neumann, as quoted by Enrico Fermi

## K2-18 b and dimethyl sulfide

* K2-18 b is a planet larger than Earth, in its star's habitable zone
* JWST's MIRI instrument measured starlight passing through its atmosphere - a bump in the spectrum means a molecule absorbing at that wavelength
* DMS, dimethyl sulfide, is made on Earth almost entirely by marine life

<img src="images/madhusudhan2025_fig2_k218b_miri.png" alt="JWST MIRI transmission spectrum of K2-18 b with red error bars, a blue model with shaded bands, and labels DMS, DMDS under two broad bumps" style="display:block;margin:0 auto;max-height:195px">

<div style="font-size:0.75em; text-align:center; color:#666;">Madhusudhan et al. 2025, ApJL 983, L40 (arXiv:2504.12267), Fig. 2: the MIRI spectrum, the retrieved model and molecule labels.</div>

<div style="font-size:1.15em; line-height:1.4; margin:0.6em 0; padding:0.5em 0.8em; border-left:4px solid #444; background:#f2f2f2;">

💬 look only at the red points. Do you see the features the labels point at? Yes or no in chat.

</div>

**Judge the red points on their own, then the model curve.**

## K2-18 b: a flat line

<img src="images/taylor2025_fig1_k218b_flatline.png" alt="The same K2-18 b points, in parts per million, fitted with a flat line, chi-squared per degree of freedom 1.06, and with a single Gaussian bump" style="display:block;margin:0 auto;max-height:260px">

<div style="font-size:0.75em; text-align:center; color:#666;">Taylor 2025, arXiv:2504.15916, Fig. 1 (top two panels): a flat line and a single Gaussian fitted to the same points.</div>

* a flat line, no molecule at all, fits with $\chi^2/\nu = 1.06$
* the ln-evidence is $-17.69$ for the flat line and $-17.64$ with a bump: $2\ln B \approx 0.1$, "weak" on Day 13's scale
* even bumps pinned at the DMS wavelengths reach only about $2\sigma$

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Always plot the null model - here a flat line - on the same axes.**

</div>

## LK-99

* a superconductor has exactly zero resistance. LK-99 (2023) seemed to reach zero near 105 °C (378 K)

<img src="images/day15_pair_lee2023_zhu2023.png" alt="LK-99 resistivity against temperature on a linear axis, appearing to fall to zero below about 105 degrees Celsius | Resistivity of copper sulfide on a linear axis, which looks like a drop to zero, and on a log axis, where it stays finite" style="display:block;margin:0 auto;max-height:187px;max-width:100%">

<div style="font-size:0.75em; text-align:center; color:#666;">Left: Lee et al. 2023, arXiv:2307.12037, Fig. 5 (LK-99, in °C). Right: Zhu et al. 2023, Matter 6, 4401 (arXiv:2308.04353), Fig. 2: copper sulfide, in K, on linear (a) and log (b) axes.</div>

* right: copper sulfide, an impurity in LK-99, drops by a factor of 1000 at 380 K. Linear axis (a): zero. Log axis (b): still finite - LK-99's "zero" was this impurity changing phase

**Zero on a linear axis only means small compared with the top of the axis.**

## Linear or log-log?

* made-up data: a broken power law, slope $-1.5$ below a break and $-3$ above it
* a power law $N \propto L^{a}$ is a straight line on log-log axes, with slope $a$
* on linear axes this looks like one steep fall. The animation moves the same points onto log-log axes, where they become two straight lines and a break

<img src="images/day15_linear_to_loglog.gif" alt="Animation: points that look like one steep fall on linear axes morph onto log-log axes, where they become two straight lines with slopes -1.5 and -3 and a break" style="display:block;margin:0 auto;max-height:340px">

**Positive quantities that span several powers of ten belong on log axes.**

## Color maps

<img src="images/crameri2020_fig1_jet_vs_batlow_images.png" alt="The Earth, a photo of Marie Curie and an apple, in the original grayscale, in jet, and in batlow: jet adds bright bands and edges that are not in the photos" style="display:block;margin:0 auto;max-height:260px">

<div style="font-size:0.75em; text-align:center; color:#666;">Crameri, Shephard &amp; Heron 2020, Nature Communications 11, 5444, Fig. 1 (CC BY 4.0): grayscale photos (a) drawn in jet (b) and in batlow (c).</div>

* a color map turns a number into a color. These photos are just numbers: brightness in each pixel
* jet puts bright bands across Marie Curie's face that are not in the photo
* in a perceptually uniform map, equal steps in the data look like equal steps in color: viridis, cividis, batlow
* jet's lightness rises and falls, which draws edges the data do not have

**Use a perceptually uniform color map.**

## Does jet add structure?

<img src="images/day15_cmap_lightness.png" alt="Lightness of three color maps against data value: viridis and gray rise steadily; jet rises to a plateau near the middle and falls again" style="display:block;margin:0 auto;max-height:220px">

<div style="font-size:0.75em; text-align:center; color:#666;">Perceived lightness ($L^*$) along each color map, computed for this course.</div>

* left: a smooth field. Right: one row of it, along the dashed line
* try jet first, then viridis, then gray - the row never changes. Then switch the data to the SDSS star counts from earlier (all six views are in one figure in the notebook, if you're reading it without running it)

In [ ]:
# A smooth field, or real SDSS star counts, drawn in a color map you pick (a "skip" cell)
_yy, _xx = np.mgrid[-3:3:300j, -3:3:300j]
CM_FIELD = np.exp(-0.5 * ((_xx - 0.4)**2 + (_yy + 0.2)**2) / 1.6) + 0.15 * _xx
CM_STARS = np.log10(_Hs.T + 1)                 # smoothed SDSS star counts from the "too many points" slide
def plot_cmap(data='smooth field', cmap='jet'):
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(10.5, 4.0), gridspec_kw=dict(width_ratios=[1.25, 1]))
    if data == 'smooth field':
        im = a1.imshow(CM_FIELD, cmap=cmap, origin='lower', extent=(-3, 3, -3, 3), aspect='auto'); row, xs = CM_FIELD[150], np.linspace(-3, 3, 300)
        a1.axhline(0, color='w', ls='--', lw=1.5); a1.set_xticks([]); a1.set_yticks([]); a2.set_xlabel('$x$')
    else:
        im = a1.imshow(CM_STARS, cmap=cmap, origin='upper', extent=(_xe[0], _xe[-1], _ye[-1], _ye[0]), aspect='auto')
        a1.set_xlim(_xe[-1], _xe[0]); j = np.searchsorted(_ye, 4.2); row, xs = CM_STARS[j], 0.5 * (_xe[1:] + _xe[:-1])
        a1.axhline(_ye[j], color='w', ls='--', lw=1.5); a1.set_xlabel('$T_{\\rm eff}$ (K)'); a1.set_ylabel('$\\log g$')
        a2.set_xlim(_xe[-1], _xe[0]); a2.set_xlabel('$T_{\\rm eff}$ (K)')
    fig.colorbar(im, ax=a1, shrink=0.85); a1.set_title(cmap, fontsize=13)
    a2.plot(xs, row, 'k', lw=2.5); a2.set_title('the row along the dashed line', fontsize=13); a2.set_ylabel('value')
    plt.tight_layout(); plt.show()

# a static copy (both datasets in jet, viridis and gray), for anyone reading without running it
if not os.path.exists('images/day15_cmap_jet_viridis_gray.png'):
    _show = plt.show; plt.show = lambda: None
    figc, axsc = plt.subplots(3, 2, figsize=(14, 6.4))
    for _r, _cm in enumerate(['jet', 'viridis', 'gray']):
        for _k, _d in enumerate(['smooth field', 'SDSS stars']):
            plot_cmap(_d, _cm); _f = plt.gcf(); _f.canvas.draw()
            axsc[_r, _k].imshow(np.asarray(_f.canvas.buffer_rgba())); axsc[_r, _k].axis('off'); plt.close(_f)
    plt.figure(figc.number); plt.tight_layout(); figc.savefig('images/day15_cmap_jet_viridis_gray.png', dpi=90, bbox_inches='tight'); plt.close(figc)
    plt.show = _show


In [ ]:
# left: the chosen data in the chosen color map; right: the row along the dashed line
controls_right(plot_cmap, data=Dropdown(options=['smooth field', 'SDSS stars'], value='smooth field', description='data'),
               cmap=Dropdown(options=['jet', 'viridis', 'cividis', 'gray', 'magma'], value='jet', description='color map'))

*Reading this without running the notebook? Here are both datasets in jet, viridis and gray:*

<img src="images/day15_cmap_jet_viridis_gray.png" alt="Six panels: a smooth field and the SDSS star counts, each in jet, viridis and gray, with the row along the dashed line beside each; only jet shows rings and bands" style="display:block;margin:0 auto;max-width:100%">

<div style="font-size:1.15em; line-height:1.4; margin:0.6em 0; padding:0.5em 0.8em; border-left:4px solid #444; background:#f2f2f2;">

💬 in jet, how many rings or bands do you see? A number in chat.

</div>

**Count the rings in jet, then look again in viridis or gray.**

## Jet vs viridis

<img src="images/day15_sdss_jet_vs_viridis.png" alt="The SDSS star counts in jet and in viridis with the row along the dashed line: jet shows a bright ring around the main sequence that viridis does not" style="display:block;margin:0 auto;max-height:260px">

<div style="font-size:0.75em; text-align:center; color:#666;">The SDSS counts from the widget, in jet (top) and viridis (bottom). The row (right) is the same in both.</div>

* the smooth row has no edges. Any ring you saw in jet came from the color map
* in the SDSS counts, jet draws a bright cyan-yellow ring around the main sequence - jet is lightest at middle values
* in a perceptually uniform map (viridis, gray) the same counts fade smoothly

**In viridis or gray the rings go away - jet adds edges that aren't in the data.**

## Visualizing to communicate

* we visualize to *explore* and to *communicate*. So far we've been exploring; the rest of today is about plots for someone else
<img src="images/lisnic2023_fig1_misleading_posts.png" alt="Three social-media posts, each a correctly drawn COVID-19 chart with a misleading claim attached: excess mortality in Mexico credited to ivermectin, Gibraltar deaths blamed on vaccination, and Swedish deaths used to say there was no pandemic" style="display:block;margin:0 auto;max-height:200px">

<div style="font-size:0.75em; text-align:center; color:#666;">Lisnic, Polychronis, Lex &amp; Kogan 2023, CHI '23, doi:10.1145/3544548.3580910, Fig. 1 (CC BY 4.0): three posts from their sample of COVID-19 charts on Twitter.</div>

* each chart is drawn correctly - the misleading part is the claim attached to it. Lisnic et al. found this is the usual way charts mislead
* so say what went in, and make everything you did to the data visible

There are three things to avoid:

**Ambiguity | Distortion | Distraction**

## What goes into the plot

<img src="images/day15_pair_freedman2021_freedman2024.png" alt="Probability distributions of the Hubble constant from ten published methods, overlapping in a broad spread | Probability distributions of the Hubble constant from only three methods, Planck, DESI and tip of the red giant branch, which overlap" style="display:block;margin:0 auto;max-height:260px;max-width:100%">

<div style="font-size:0.75em; text-align:center; color:#666;">Left: Freedman 2021, ApJ 919, 16 (arXiv:2106.15656), Fig. 10. Right: Freedman et al. 2024 (arXiv:2408.06153), Fig. 1. See Riess et al. 2024 (arXiv:2408.11770) for the other side.</div>

* $H_0$, the Hubble constant, is today's expansion rate. Early- and late-Universe measurements disagree: the Hubble tension
* Freedman 2021 (left) drew ten published methods. Freedman et al. 2024 (right) drew three: Planck, DESI, and their own JWST tip-of-the-red-giant-branch value.

**Choosing what goes into a figure is an analysis decision.**

## Ambiguity: a Rubin light curve

<img src="images/malanchev2025_fig5_dp1_eclipsing_binary.png" alt="Light curve of an eclipsing binary against phase from Rubin DP1 in six bands plus Gaia, CRTS and ATLAS points; the u-band points are labelled u minus 1 in the legend and sit near 17.9 mag" style="display:block;margin:0 auto;max-height:270px">

<div style="font-size:0.75em; text-align:center; color:#666;">Malanchev et al. 2025, arXiv:2506.23955, Fig. 5 (bottom panel), CC BY 4.0: an eclipsing binary in Rubin Data Preview 1, with Gaia, CRTS and ATLAS.</div>

* an eclipsing binary: two stars that pass in front of each other, folded on the 0.24-day period. u, g, r, i, z, y are Rubin's six filters, u the bluest

<div style="font-size:1.15em; line-height:1.4; margin:0.6em 0; padding:0.5em 0.8em; border-left:4px solid #444; background:#f2f2f2;">

💬 what is this star's u magnitude near phase 0.25? A number in chat.

</div>

## The u−1 shift

<img src="images/day15_rubin_u_shift_annotated.png" alt="The Rubin light curve with the u-band points circled in red and labelled u minus 1, really 1 mag fainter" style="display:block;margin:0 auto;max-height:240px">

<div style="font-size:0.75em; text-align:center; color:#666;">Malanchev et al. 2025, Fig. 5 (CC BY 4.0), with the u−1 points circled for this course.</div>

* the blue points are labeled u−1: each u magnitude had 1 subtracted to fit on the axis. Near phase 0.25 the star is at u ≈ 18.9 (it's plotted at 17.9)
* the shift is easy to miss: it is only in the legend and the caption
* the figure uses twelve symbols from four surveys, several in nearly the same color
* magnitudes are backwards (brighter is up) - astronomers expect that, but label it for anyone else

**Write out explanations of the data on the graph itself.** (Tufte)

## Distortion: a state COVID-19 chart

* a state health department's chart of daily COVID-19 cases in five counties, May 2020
* read the dates along the bottom, left to right

<img src="images/georgia_dph_2020_covid_chart.png" alt="Bar chart from the Georgia Department of Public Health, May 2020, of daily COVID-19 cases in five counties, with dates along the bottom" style="display:block;margin:0 auto;max-height:280px">

<div style="font-size:0.75em; text-align:center; color:#666;">Georgia Department of Public Health, posted May 10 2020; via Route Fifty, May 18 2020.</div>

<div style="font-size:1.15em; line-height:1.4; margin:0.6em 0; padding:0.5em 0.8em; border-left:4px solid #444; background:#f2f2f2;">

💬 the chart seems to show cases falling. What is problematic? One line in chat.

</div>

**Read the axis before you read the trend.**

## The dates were out of order

<img src="images/day15_dates_order_demo.png" alt="Two bar charts of the same 14 made-up daily counts: sorted by count they fall steadily, in date order they rise" style="display:block;margin:0 auto;max-height:220px">

<div style="font-size:0.75em; text-align:center; color:#666;">Made-up counts, made for this course: the same 14 numbers in both panels.</div>

* the dates ran out of time order, in an order that made the counts fall, and the bars in each day were sorted largest first
* the department fixed it the next day

**The order of an axis is part of the figure.**

## Distortion: an election-night chart

<img src="images/fivethirtyeight2020_wisconsin_cumulative.png" alt="FiveThirtyEight chart of total Republican and Democratic votes counted in Wisconsin on election night 2020, against time; the blue line jumps vertically before 6 a.m." style="display:block;margin:0 auto;max-height:330px;max-width:100%">

<div style="font-size:0.75em; text-align:center; color:#666;">FiveThirtyEight / ABC News, 6:23 a.m. Nov 4 2020, as screenshotted by BOOM Live; reproduced for teaching.</div>

<div style="font-size:1.15em; line-height:1.4; margin:0.6em 0; padding:0.5em 0.8em; border-left:4px solid #444; background:#f2f2f2;">

💬 posts shared this as proof that over 100,000 ballots were "dumped" overnight. What does the vertical jump actually show? One line in chat.

</div>

## Election night: what the jump shows

* the y-axis is the running total of votes *counted*, and the x-axis is when they were *reported*. A vertical step is one county releasing a batch at once - here Milwaukee's absentee ballots
* the batch was not 100% Biden: Trump's red line also rises, hidden behind the blue one. And one tick reads "5k" where it should read 500k
* PolitiFact rated the fraud claim False

<div style="font-size:0.75em; text-align:center; color:#666;">PolitiFact, Nov 4 2020, "No, these FiveThirtyEight graphs don't prove voter fraud".</div>

**A cumulative count against reporting time shows when results arrived, not when votes were cast.**

## Distortion: two bars

<img src="images/whitehouse2025_encounters_95pct.jpg" alt="White House graphic headlined Daily illegal alien encounters plummet 95 percent under President Trump, with a tall red bar of 20,086 for Biden and a short green bar of 1,041 for Trump" style="display:block;margin:0 auto;max-height:340px;max-width:100%">

<div style="font-size:0.75em; text-align:center; color:#666;">White House, Instagram, Feb 15 2025; reproduced for teaching.</div>

<div style="font-size:1.15em; line-height:1.4; margin:0.6em 0; padding:0.5em 0.8em; border-left:4px solid #444; background:#f2f2f2;">

💬 the title says "daily". Is the tall bar a daily number? One line in chat.

</div>

## Two bars: mixed units

* 20,086 is the *total* for Biden's last seven days in office. 1,041 is the *daily average* for Trump's first seven
* per day, that is 2,869 against 1,041 - a 60% drop, not 95%
* both bars carry the same axis, so the reader assumes the same units. Encounters also count events, not people: one person crossing twice is two

<div style="font-size:0.75em; text-align:center; color:#666;">PolitiFact, Feb 20 2025, "The White House exaggerated how sharply illegal immigration fell".</div>

**Bars on one axis must be in the same units.**

## Distortion: the temperature axis

<img src="images/day15_gistemp_0_110F.png" alt="Global mean temperature from 1880 to 2025 on an axis from -10 to 110 degrees Fahrenheit: a flat line near 57 to 59 degrees" style="display:block;margin:0 auto;max-height:300px;max-width:100%">

<div style="font-size:0.75em; text-align:center; color:#666;">Redrawn for this course from NASA GISTEMP v4 (global annual mean, anomaly + 14 °C as an illustrative absolute) in the style of a National Review tweet, Dec 14 2015: "The only #climatechange chart you need to see."</div>

<div style="font-size:1.15em; line-height:1.4; margin:0.6em 0; padding:0.5em 0.8em; border-left:4px solid #444; background:#f2f2f2;">

💬 is this chart wrong? Is it misleading? Answer both in chat.

</div>

## Climate: the axis

<img src="images/day15_gistemp_axis_range.png" alt="The same NASA temperature data on a -10 to 110 F axis (flat) and as an anomaly on an axis scaled to the change (a rise of about 1.4 C since 1880)" style="display:block;margin:0 auto;max-height:240px;max-width:100%">

<div style="font-size:0.75em; text-align:center; color:#666;">NASA GISTEMP v4, course redraw. Critique of the 2015 tweet: Halpern, Union of Concerned Scientists blog, Dec 15 2015.</div>

* not wrong - the numbers are right. Misleading: the axis spans 120 °F for a change of about 1.4 °C (2.5 °F), so any change looks flat
* choose the range for the change you are asking about

**The axis range is an analysis choice.**

## Climate: the fit window

<img src="images/skepticalscience_escalator_pair.png" alt="Two copies of the same monthly temperature series, 1970-2014: on the left fitted in short segments that each look flat, on the right with one rising trend line" style="display:block;margin:0 auto;max-height:260px;max-width:100%">

<div style="font-size:0.75em; text-align:center; color:#666;">Skeptical Science, "The Escalator" (CC BY), NASA GISS data, 1970-2014.</div>

* left: the same data fitted in short segments - each looks flat, so "warming has stopped" again and again
* right: one fit over the whole period rises. Choosing the window is choosing the answer

**Fit over a window long enough to average out the year-to-year noise.**

## Distortion: two y-axes

* a chart from the UK Office for National Statistics (ONS), on its own blog about why dual-axis charts mislead: employment rate (left axis) and unemployment rate (right axis)

<img src="images/ons2019_dual_axis_employment.png" alt="UK employment rate on the left axis from 68 to 76 percent and unemployment rate on the right axis from 2 to 10 percent, 2006 to 2018; the two lines cross near 2008 and 2014" style="display:block;margin:0 auto;max-height:290px">

<div style="font-size:0.75em; text-align:center; color:#666;">UK Office for National Statistics, Digital blog, "Dueling with axis: the problems with dual axis charts" (Donnarumma, July 2019), Open Government Licence v3.0.</div>

<div style="font-size:1.15em; line-height:1.4; margin:0.6em 0; padding:0.5em 0.8em; border-left:4px solid #444; background:#f2f2f2;">

💬 what happened in 2008 and 2014, when the lines cross? One line in chat.

</div>

## Two y-axes

<img src="images/ons_fix_split_8.png" alt="The ONS fix: two stacked panels, employment rate 70 to 76 percent on top and unemployment rate 4 to 10 percent below, sharing one time axis" style="display:block;margin:0 auto;max-height:230px">

<div style="font-size:0.75em; text-align:center; color:#666;">Source: Office for National Statistics (Donnarumma 2019), Open Government Licence v3.0: their fix, two charts.</div>

* nothing happened - the lines cross because of where the two axes were placed. 72% employment and 6% unemployment share one height
* shift either axis and the crossings move, or vanish
* the ONS's own fixes: one axis with labels on the chart, or separate the chart into two

**Use one axis, or two charts.**

## Distortion: log axes and the public

<img src="images/day15_pair_romano2020_day15.png" alt="The same US COVID-19 total deaths, February to April 2020, on a linear axis (a steep hockey stick) and on a log axis (a curve that flattens) | Reminder from earlier: the broken power law on log-log axes" style="display:block;margin:0 auto;max-height:183px;max-width:100%">

<div style="font-size:0.75em; text-align:center; color:#666;">Left: the graphs Romano et al. 2020 showed (Health Economics, doi:10.1002/hec.4143, CC BY 4.0; data and graphs from worldometers.info). Right: our log-log animation from earlier.</div>

* earlier: positive quantities that span powers of ten belong on log axes - fine for astronomers
* Romano et al. 2020 showed 2,000 US residents the same COVID-19 death counts on a log or a linear axis
* 41% answered a question about the log graph correctly, against 84% for the linear graph
* asked for total deaths one week later, the log group guessed 71,250 and the linear group 63,429. The actual number was 54,256

<div style="font-size:0.75em; text-align:center; color:#666;">Romano, Sotis, Dominioni &amp; Guidi 2020, Health Economics, doi:10.1002/hec.4143; numbers from the authors' LSE COVID-19 blog post, May 19 2020.</div>

**Choose the axis for the reader: astronomers read log axes, most people don't.**

## Distraction: a 3D pie

<img src="images/usb2013_soybean_meal_3dpie.jpg" alt="A tilted 3D pie chart of global soybean meal consumption by region on a world-map background, with a globe icon, an ornate frame, a logo banner, and eight thin slices with leader lines" style="display:block;margin:0 auto;max-height:330px">

<div style="font-size:0.75em; text-align:center; color:#666;">United Soybean Board, "Global Soybean Meal Consumption by Region (2012/13)", 2013; data USDA. CC BY 2.0, via Wikimedia Commons.</div>

<div style="font-size:1.15em; line-height:1.4; margin:0.6em 0; padding:0.5em 0.8em; border-left:4px solid #444; background:#f2f2f2;">

💬 which is bigger, South America or North America? And which of the 1% regions is largest? One line in chat.

</div>

## Distraction: the same numbers, redrawn

<img src="images/day15_soybean_redrawn.png" alt="The same 13 regional shares as a sorted horizontal bar chart: East Asia 32 percent, North America 18, European Union 16, South America 12, Southeast Asia 8, then 3, 3, 2, 2 and four regions at 1 percent" style="display:block;margin:0 auto;max-height:290px">

<div style="font-size:0.75em; text-align:center; color:#666;">The percentages printed on the pie, redrawn as a sorted bar chart for this course.</div>

* North America (18%) is half again South America (12%) - the tilt makes the front slice look as big. The four 1% regions are tied
* the map, globe, frame, banner, 3D tilt and 15 colors carry no data. They take most of the ink and hide the comparisons

**Push up the data-ink ratio: the share of the ink that shows data.** (Tufte)

## Color for readers

<img src="images/crameri2020_fig2_colourblind.png" alt="Color maps as seen with three kinds of color-vision deficiency and in grayscale: viridis, cividis and batlow stay ordered, jet does not" style="display:block;margin:0 auto;max-height:230px">

<div style="font-size:0.75em; text-align:center; color:#666;">Crameri, Shephard &amp; Heron 2020, Nature Communications 11, 5444, Fig. 2 (CC BY 4.0).</div>

* about 1 in 12 men has a color-vision deficiency, mostly red-green
* add markers, line styles and labels so a figure works in black and white
* a diverging map uses one color below a middle value, another above, and white or gray in between: good when the middle means something

**Check how your figure reads in grayscale and to color-blind readers.**

## The color map changed the diagnosis

<img src="images/borkin2011_fig1_rainbow_vs_diverging.png" alt="Artery shear stress shown in a rainbow map (left, 2D and 3D) and a diverging red-white-black map (right, 2D and 3D)" style="display:block;margin:0 auto;max-height:250px">

<div style="font-size:0.75em; text-align:center; color:#666;">Borkin et al. 2011, IEEE TVCG 17, 2479, Fig. 1: rainbow (A, C) and diverging (B, D) maps of the same quantity.</div>

* 21 medical students looked for regions of low shear stress in arteries, a sign of heart-disease risk
* with a diverging map instead of rainbow they found about 47% more of those regions in 2D, and about 82% more in 3D

**The color map changed what they found.**

## The same black hole, four color maps

* the Event Horizon Telescope (EHT) imaged the ring of light around the black hole in M87 in 2019

<img src="images/ochigame2026_fig15_m87_four_colormaps.jpg" alt="The Event Horizon Telescope image of M87* in four color maps: viridis, a blue map, grayscale and a rainbow map" style="display:block;margin:0 auto;max-height:280px">

<div style="font-size:0.75em; text-align:center; color:#666;">Ochigame, Skulberg &amp; van Dongen 2026, arXiv:2607.14928, Fig. 15: the 2019 EHT image of M87* recolored. The EHT published it in orange on black.</div>

<div style="font-size:1.15em; line-height:1.4; margin:0.6em 0; padding:0.5em 0.8em; border-left:4px solid #444; background:#f2f2f2;">

💬 which panel would you publish, and why? One line in chat.

</div>


## Which color map?

<img src="images/eht_m87_eso1907a.jpg" alt="The Event Horizon Telescope image of M87*: an orange ring of light around a dark center, brighter at the bottom" style="display:block;margin:0 auto;max-height:220px">

<div style="font-size:0.75em; text-align:center; color:#666;">The image as the EHT published it, 2019. Credit: EHT Collaboration (ESO eso1907a, CC BY 4.0).</div>

* there isn't one right answer - but viridis and gray are perceptually uniform, and the rainbow panel adds bands that aren't in the data (as jet did earlier). Maybe acceptable. Depends on what you are trying to show

**The data are the same in all four panels. The color map is a choice you have to defend.**

## Gravitational waves: GWTC-4

<img src="images/ligo_hanford_aerial.jpg" alt="Aerial view of the LIGO Hanford detector: a corner building in desert scrub with two long arm tubes at right angles" style="display:block;margin:0 auto;max-height:200px">

<div style="font-size:0.75em; text-align:center; color:#666;">LIGO Hanford, one of the two LIGO detectors. Courtesy Caltech/MIT/LIGO Laboratory.</div>

* GWTC-4: the catalog of gravitational-wave events from LIGO, Virgo and KAGRA - merging black holes and neutron stars. Chirp mass $\mathcal{M}$ sets how fast the signal sweeps up in frequency, and $q$ is the mass ratio
* each event's analysis gives a posterior for every parameter: masses, spins, distance

**How would you compare dozens of posteriors at once?**

## Small multiples: GWTC-4

<img src="images/lvk2025_gwtc4_fig2_small_multiples_crop.png" alt="Grid of posterior distributions, one row per gravitational-wave event and one column per parameter: chirp mass, mass ratio, two spins and distance, each drawn as a two-sided violin in a different color" style="display:block;margin:0 auto;max-height:230px">

<div style="font-size:0.75em; text-align:center; color:#666;">LIGO-Virgo-KAGRA Collaboration 2025, GWTC-4.0, arXiv:2508.18082, Fig. 2 (first 16 rows), CC BY 4.0.</div>

* one row per gravitational-wave event, one column per parameter, and the same axes in every row. This layout is called **small multiples**, and it lets you compare dozens of events quickly
* each event keeps its own color in every column, so you can follow one event across the panels and tell neighboring rows apart
* each shape has two halves - colored with the default prior, white recomputed assuming the population of black holes the whole catalog implies - explained only in the caption (the ambiguity here)

**Same axes in every panel, and the same color for the same event.**

## Dark energy: DESI DR2

<img src="images/desi_mayall_night_noirlab.jpg" alt="The Mayall 4-m telescope dome on Kitt Peak at dusk under a starry sky; DESI sits on this telescope" style="display:block;margin:0 auto;max-height:200px">

<div style="font-size:0.75em; text-align:center; color:#666;">The Mayall 4-m telescope on Kitt Peak, home of DESI. Credit: KPNO/NOIRLab/NSF/AURA/M. Chung (LBNL), CC BY 4.0.</div>

* DESI maps millions of galaxies to measure how the Universe expanded. With the CMB and supernovae (Pantheon+, Union3, DES Y5 - my field) it constrains dark energy
* $w$ is dark energy's pressure over its density, $a$ the size of the Universe relative to today: $w(a) = w_0 + w_a(1-a)$. A cosmological constant is $w_0 = -1$, $w_a = 0$, where the gray dashed lines cross

**Is $w$ constant at $-1$?**

## DESI DR2: the figure

<img src="images/desi2025_fig11_w0wa_contours.png" alt="Contours in the w0-wa plane for DESI plus CMB combined with three supernova samples, Pantheon+, Union3 and DES Y5, with DESI plus CMB alone dashed; dashed gray lines mark w0 = -1 and wa = 0" style="display:block;margin:0 auto;max-height:260px">

<div style="font-size:0.75em; text-align:center; color:#666;">DESI Collaboration 2025, DESI DR2 Results II, arXiv:2503.14738, Fig. 11, CC BY 4.0.</div>

* a cosmological constant is $w_0 = -1$, $w_a = 0$, where the gray dashed lines cross
* a single panel, with a color for each supernova sample. The model being tested is drawn, so you see how far the contours sit from it without reading a number

<div style="font-size:1.15em; line-height:1.4; margin:0.6em 0; padding:0.5em 0.8em; border-left:4px solid #444; background:#f2f2f2;">

💬 one thing you would change. One line in chat.

</div>


## DESI DR2: what it shows

<img src="images/day15_desi_annotated.png" alt="The DESI w0-wa figure with two red circles: the constant-w crossing in the top-left corner and the DESI+CMB contour running off the bottom right" style="display:block;margin:0 auto;max-height:240px">

<div style="font-size:0.75em; text-align:center; color:#666;">DESI Collaboration 2025, Fig. 11 (CC BY 4.0), with the two things to change circled for this course.</div>

* what it shows: the contours sit away from the crossing - DESI+CMB+supernovae prefer an evolving $w$ over a constant at 2.8-4.2σ, depending on which supernova sample is used
* things you might change: the constant sits in the top-left corner of the axes, and the DESI+CMB contour runs off the bottom of the plot

**Draw the model you are testing against.**

## Tufte's rules

* The representation of numbers, as physically measured on the surface of the graph itself, should be directly proportional to the numerical quantities represented
* Clear, detailed and thorough labeling should be used to defeat graphical distortion and ambiguity. Write out explanations of the data on the graph itself. Label important events in the data.
* Show data variation, not design variation.
* In time-series displays of money, deflated and standardized units of monetary measurement are nearly always better than nominal units.
* The number of information carrying (variable) dimensions depicted should not exceed the number of dimensions in the data. Graphics must not quote data out of context.

<div style="font-size:0.75em; text-align:center; color:#666;">Tufte 1983, <i>The Visual Display of Quantitative Information</i>.</div>

## Wrapping up

You should be able to:

* explain why summary statistics hide the shape of the data, and plot the density when points overlap
* read residuals, trace plots and $\tau$ - and say what parallel tempering does
* run a posterior predictive check (Day 9's Cepheids)
* spot the figures that fooled people (Venus, K2-18 b, LK-99) and choose log axes and color maps well
* make plots for other people: say what went in, label shifts on the figure, use one y-axis, the same axes across panels, and draw the reference model

<img src="images/day15_wrap_montage.png" alt="A strip of six figures from today: the Datasaurus, Anscombe residuals, the white-dwarf trace, the Cepheid test statistic, the jet versus batlow photos and the DESI contours" style="display:block;margin:0 auto;max-height:90px">

<div style="font-size:0.75em; text-align:center; color:#666;">Today, left to right: Datasaurus, Anscombe residuals, trace plots, the Cepheid check, color maps, DESI.</div>

**Plot the data, the residuals, and fake data from the model, and ask what the plot left out.**

Reading: VanderPlas, *Python Data Science Handbook*, Ch. 4, free at jakevdp.github.io/PythonDataScienceHandbook

Tuesday, Oct 20, in person: if the mantra of this class is looking at your data, then it's important to present your data for others to look at effectively

## Tue Oct 20, 3:45 pm, Astronomy 134: Samaresh Mondal (Illinois)

<div style="display:flex; align-items:flex-start; gap:1.4em; margin-top:0.4em;">
<img src="images/speaker_samaresh_mondal.jpg" alt="Photo of Samaresh Mondal" style="max-width:190px; border-radius:4px;">
<div>

* **"Characterizing Tidal Disruption Events and AGN Variability"**
* a tidal disruption event (TDE) is a star torn apart by a supermassive black hole. Active galactic nuclei (AGN) also flare, so a flare alone can't tell you which
* Mondal and French (2025) measure how much AGN X-ray light curves vary with time lag, and ask which TDE candidates vary more than an AGN can
* light curves and variability are where we go next: Lab 07 is time series and Gaussian processes
* refreshments from 3:30 pm in Astronomy 222

</div>
</div>

<div style="font-size:0.75em; text-align:center; color:#666;">Listing: astro.illinois.edu/news-events/astrophysics-colloquium (checked 2026-10-15). Photo: Illinois Astronomy directory, used to advertise his own talk. Paper: Mondal &amp; French 2025, arXiv:2508.03889.</div>

## EXTRA: A retracted superconductivity claim

<img src="images/vandermarel2022_fig1_dias_signal.png" alt="Two curves on two y axes for six pressures: the measured voltage and the claimed superconducting signal, which look almost identical" style="display:block;margin:0 auto;max-height:290px">

<div style="font-size:0.75em; text-align:center; color:#666;">van der Marel &amp; Hirsch 2022, IJMPB 37, 2375001 (arXiv:2201.07686), Fig. 1, on Snider et al. 2020, Nature 586, 373 (retracted: Nature 610, 804, 2022).</div>

* the 2020 Nature paper claimed room-temperature superconductivity under pressure, from a magnetic signal with a background subtracted
* van der Marel and Hirsch found the "raw" data looked like the final curve plus noise - the reverse of the stated procedure

**Raw data should look noisier than the final result.**

## EXTRA: A plateau that depended on the range

<img src="images/day15_pair_zhang2021_zhang2021.png" alt="Conductance of a nanowire device at 0.8 tesla: a plateau near the quantized value over part of the gate-voltage range | The same device at 0.7 tesla: the conductance overshoots the quantized value" style="display:block;margin:0 auto;max-height:260px;max-width:100%">

<div style="font-size:0.75em; text-align:center; color:#666;">Zhang et al. 2021, arXiv:2101.11456, Figs. 3 and 5, posted with the retraction of Zhang et al. 2018, Nature 556, 74 (Nature 591, E30, 2021).</div>

* the 2018 paper claimed conductance quantized at exactly $2e^2/h$ (the quantum of conductance), a sign of an exotic particle (a Majorana mode)
* in the full data the same device sits near $2e^2/h$ at 0.8 T and overshoots it at 0.7 T
* the 2018 figure showed the gate-voltage range where it looked quantized

**Show the full range, or say why you cut it.**